# 03 · LLM backends — one interface, four models

**Step 3 of the Study Assistant project.** The assistant needs a language model to write answers from the retrieved course material. We support four, behind the same `generate(prompt, system=...)` call (`src/study_assistant/llm/`):

| Backend | Model (default) | Runs | Cost | Needs |
|---|---|---|---|---|
| `openai` | `gpt-5.6-luna` | cloud | paid (cheap) | `OPENAI_API_KEY` in `.env` |
| `gemini` | `gemini-3.6-flash` | cloud | free tier | `GEMINI_API_KEY` in `.env` ([get one](https://aistudio.google.com)) |
| `ollama` | `llama3.1:8b` | your laptop | free | [Ollama](https://ollama.com) + `ollama pull llama3.1:8b` |
| `flan` | `google/flan-t5-base` | your laptop | free | nothing (downloads ~1 GB once) |

Defaults can be changed in `.env` (`OPENAI_MODEL`, `GEMINI_MODEL`, `OLLAMA_MODEL`, `FLAN_MODEL`). All calls use `temperature=0` so evaluation runs are reproducible, and rate-limit errors are retried with backoff (important for Gemini's free tier).

From the terminal: `uv run python -m study_assistant.llm --check`

In [1]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_colwidth", 120)
from study_assistant.llm import available_backends, get_llm, BACKENDS

status = available_backends()
pd.Series(status, name="status").to_frame()

,status
openai,unavailable: OPENAI_API_KEY is not set. Add it to .env (see .env.example).
gemini,unavailable: GEMINI_API_KEY is not set. Get a free key at https://aistudio.google.com and add it to .env.
ollama,ok (llama3.1:8b)
flan,ok (google/flan-t5-base)


## Same question, with and without the course material

This is the core idea of the project in miniature. *Without* context the model answers from memory; *with* context it answers from the retrieved course chunks (Step 2).

In [2]:
from study_assistant.retrieval import build_retriever

question = "What is the reparameterization trick in a VAE?"
hits = build_retriever("dense").search(question, k=3)
context = "\n\n".join(f"[{i}] {h.chunk.text}" for i, h in enumerate(hits, 1))
for h in hits:
    print(h.chunk.citation)

no_context = f"Question: {question}\nAnswer in two sentences."
with_context = f"Answer the question using only the context below, in two sentences.\n\nContext:\n{context}\n\nQuestion: {question}\nAnswer:"

W1124 16:00:48.206000 73094 /Users/apple/Desktop/ /Columbia/Sem 3/Assignments/Applied Gen AI/Final Project/.venv.nosync/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13282.09it/s]

Module 4: Practical - Variational Autoencoders, cells 33-35 (Variational Autoencoder (VAE) Architecture)
Module 4: Practical - Variational Autoencoders, cells 36-38 (Variational Autoencoder (VAE) Architecture › We also need to slightly change the loss function.  In addi…)
Module 4: Practical - Variational Autoencoders, cells 33-35 (Variational Autoencoder (VAE) Architecture)


In [3]:
rows = []
for b in BACKENDS:
    if not status[b].startswith("ok"):
        continue
    llm = get_llm(b)
    for label, prompt in [("no context", no_context), ("with course context", with_context)]:
        r = llm.generate(prompt, max_tokens=200)
        rows.append({"backend": b, "model": r.model, "prompt": label, "seconds": r.latency_s,
                     "tokens in/out": f"{r.input_tokens}/{r.output_tokens}", "answer": r.text})
results = pd.DataFrame(rows)
results[["backend", "prompt", "seconds", "tokens in/out"]]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 6161.64it/s]

,backend,prompt,seconds,tokens in/out
0,ollama,no context,4.190,29/68
1,ollama,with course context,4.063,260/67
2,flan,no context,2.642,22/9
3,flan,with course context,0.202,297/7


In [4]:
for r in rows:
    print(f"### {r['backend']} — {r['prompt']}\n{r['answer']}\n")

### ollama — no context
The reparameterization trick is a technique used in Variational Autoencoders (VAEs) to allow for the backpropagation of gradients through the stochastic sampling process in the encoder. It involves transforming the stochastic latent variable into a deterministic one by sampling from a fixed distribution, such as a normal distribution, conditioned on the input data.

### ollama — with course context
The reparameterization trick in a VAE is a method used to sample from the distribution output by the encoder, rather than directly sampling from the encoder itself. This is done by the `reparameterize` function, which takes the mean and log variance of the normal distribution output by the encoder and returns a sampled latent variable.

### flan — no context
a reversible circuit

### flan — with course context
map to a distribution



**What to notice**

- **Llama 3.1 without context** gives a generic textbook answer. It happens to be right here, but it is not reliable: asked the same thing with a slightly different prompt while building this step, it said the trick turns a *discrete* latent variable into a continuous one, which is wrong for VAEs. Answers from memory also can't be cited and never say "not covered".
- **Llama 3.1 with the course notebook as context** answers in the course's own terms (the `reparameterize(mu, logvar)` function from the Lecture 5 practical), and the answer can be traced to a source.
- **flan-t5** cannot answer from memory at all and gives very short answers even with context. It reads at most 512 tokens, so long contexts are cut. It stays in the project as the *weak baseline* (and because it is the model used in the class RAG notebooks).
- OpenAI / Gemini rows appear here once their keys are added to `.env`.

Whether answers are *correct and properly cited* is measured on 50 questions in Step 5, not judged from one example.